# Generation Data Loading Test
This notebook tests the loading of the generation dataset and inspects the data format.

In [4]:
import sys
import os
import pprint
import torch
from transformers import AutoTokenizer

# Add root directory to path to import project modules
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_dir not in sys.path:
    sys.path.append(root_dir)

from ZGeneration.data_loader_gen import GenerationDataset, load_gen_data
from config_llama3 import TrainingConfig

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
"""
Original record json file didnt involve context, hence to get it back without rerun the model, we load original codebase to trace the data.
"""
sys.path.append('../')
import pickle

data_dir = r'../data/gen_task'

In [8]:
def load_gen_data(data_dir):
    cache_file = f"{data_dir}/processed_gen_data.pkl"
    if not os.path.exists(cache_file):
        raise FileNotFoundError(f"Generation data not found: {cache_file}")
    
    print(f"Loading Gen Data: {cache_file}")
    with open(cache_file, "rb") as f:
        data, max_len = pickle.load(f)
    print(f"Data Loaded: {len(data['input_text'])} samples.")
    return data, max_len

In [9]:
data, max_len = load_gen_data(data_dir)
print("type of data: {}".format(type(data)))

Loading Gen Data: ../data/gen_task/processed_gen_data.pkl
Data Loaded: 51239 samples.
type of data: <class 'dict'>


In [22]:
index_dict = {i: 0 for i in range(data['ud_idx'][-1])}
length = len(data['ud_idx'])
previous_ud_idx, previous_ud_idx2true_idx, ld_idx_list =0, 0, list()
for idx in range(length):
    if data['ud_idx'][idx] == previous_ud_idx:
        ld_idx_list.append(data['ld_idx'][idx])
    else:
        previous_ud_idx2true_idx += len(ld_idx_list)
        
        assert previous_ud_idx+1 == data['ud_idx'][idx]
        
        index_dict[previous_ud_idx+1] = previous_ud_idx2true_idx
        previous_ud_idx = data['ud_idx'][idx]
        ld_idx_list = [data['ld_idx'][idx]]

In [26]:
index_dict[24829] == length - 2 # 2 represent total utterance length for last dialogue

True

In [35]:
print([msg['content'] for msg in data['input_text'][-4][1:-1]])

["Situation: had to cancel our family vacation coming up next month . my husband 's work said he could not go after they already approved the time off .\n\nUser Word: i had to cancel our family vacation coming up next month ."]


In [2]:
# Configuration
data_path = os.path.join(root_dir, 'data', 'gen_task')
print(f"Data Path: {data_path}")

# Load Tokenizer
# We try to load the tokenizer defined in config, or fallback to a default if unavailable
model_name = TrainingConfig.model_name
print(f"Model Name from Config: {model_name}")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("Tokenizer loaded successfully.")
except Exception as e:
    print(f"Warning: Could not load tokenizer '{model_name}'.")
    print(f"Error: {e}")
    print("Setting tokenizer to None. Dataset __getitem__ will fail, but raw data inspection is possible.")
    tokenizer = None

Data Path: /home/new/Documents/QSH/Llama_train/data/gen_task
Model Name from Config: Llama-3.3-8B-Instruct
Error: Llama-3.3-8B-Instruct is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`
Setting tokenizer to None. Dataset __getitem__ will fail, but raw data inspection is possible.


In [3]:
# Load Data
try:
    data, max_len = load_gen_data(data_path)
    print(f"Max Length in Data: {max_len}")
    
    # Initialize Dataset
    # pass max_seq_len from config or default
    dataset = GenerationDataset(data, tokenizer, max_seq_len=TrainingConfig.hidden_size if hasattr(TrainingConfig, 'hidden_size') else 512)
    print(f"Dataset Initialized. Size: {len(dataset)}")
    
except Exception as e:
    print(f"Error loading data: {e}")
    # Create dummy data for visualization if loading fails? No, better to stop.
    raise e

Loading Gen Data: /home/new/Documents/QSH/Llama_train/data/gen_task/processed_gen_data.pkl
Data Loaded: 51239 samples.
Max Length in Data: 2195
Dataset Initialized. Size: 51239


In [ ]:
# Inspect First 10 Samples
print("Inspecting first 10 samples from the dataset (Raw Data)...\n")

for i in range(10):
    if i >= len(dataset):
        break
        
    print(f"====== Sample {i+1} ======")
    
    # Access raw data directly to avoid tokenization errors if tokenizer is missing
    # dataset.data is the dictionary structure
    input_text = dataset.data['input_text'][i]
    target_text = dataset.data['target_text'][i]
    emotion = dataset.data['emotion'][i]
    unique_idx = dataset.data['ud_idx'][i]
    local_idx = dataset.data['ld_idx'][i]
    
    print(f"Emotion: {emotion}")
    print(f"Dialogue ID: {unique_idx}, Turn ID: {local_idx}")
    print("\n[Input (Messages)]:")
    pprint.pprint(input_text, width=120)
    
    print("\n[Target Output]:")
    pprint.pprint(target_text)
    
    print("\n")

In [36]:
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu

In [37]:
def write2records(givenList, givename, givenfolder: str = '../outputs/topk/'):
    os.makedirs(givenfolder, exist_ok=True)
    
    output_path = os.path.join(givenfolder, f"{givename}.txt")

    with open(output_path, 'w', encoding='utf-8') as f:
        for i, item in enumerate(givenList):
            ud_idx, ld_idx = item['ud_idx'], item['ld_idx']
            metrics, ctx = item['metrics'], item['context']
            
            f.write(f"Unique Idx-Internal Idx: {ud_idx}-{ld_idx}\n")
            f.write(f"B1: {metrics['bleu-1']*100 :.2f}\n")
            f.write(f"B2: {metrics['bleu-2']*100 :.2f}\n")
            f.write(f"Generated: {item['generated']}\n")
            f.write(f"Reference: {item['target']}\n")
            f.write(f"Context: {ctx}\n")
            
            if i < len(givenList) - 1:
                f.write(f"{'='*70}\n")
    
    print(f"Written {len(givenList)} entries to {output_path}")

In [38]:
import os, json
read_file = [
    '../outputs/llama3.1Gen_03-06/method_SSP_bs_2_inputdata_input_text_Llama3.1_SSL02/eval_logs/eval_results_FSL_4shots_last_test.jsonl',
    '../outputs/llama3.1Gen_02-24/method_FSL_bs_2_inputdata_input_text_Llama3.1_FSL16/eval_logs/eval_results_FSL_16shots_last_test.jsonl'
]

def compare_gen(given):
    metrics = given['metrics']
    b1, b2 = metrics['bleu-1'], metrics['bleu-2']
    return b1 + b2


for filename in read_file:
    storage_list = list()
    name = '_'.join(filename.split('/')[-3].split('_')[-2:])
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            gen_word, ref_word = item['generated'], item['target']
            gen_tok, ref_tok = word_tokenize(gen_word), word_tokenize(ref_word)
            ud_idx, ld_idx = int(item['ud_idx']), int(item['ld_idx'])
            ctx = data['input_text'][index_dict[ud_idx]+ld_idx][1:-1] # ignore system and target ctx
            
            ctx = [msg['content'] for msg in ctx]
            
            b1 = sentence_bleu([ref_tok], gen_tok, weights = (1,0,0,0))
            b2 = sentence_bleu([ref_tok], gen_tok, weights=(0.5, 0.5, 0,0))
            
            item['metrics']['bleu-1'] = b1
            item['metrics']['bleu-2'] = b2
            item['context'] = ctx
            
            storage_list.append(item)
    logs_sorted = sorted(storage_list, key=compare_gen, reverse=True)
    topk100 = logs_sorted[:100]
    bad_topk100 = logs_sorted[-100:]
    
    write2records(topk100, name+'_top100')
    write2records(bad_topk100, 'bad_'+name+'_top100')

/home/new/miniconda3/lib/python3.13/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/new/miniconda3/lib/python3.13/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/new/miniconda3/lib/python3.13/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use Smoothi

Written 100 entries to ../outputs/topk/Llama3.1_SSL02_top100.txt
Written 100 entries to ../outputs/topk/bad_Llama3.1_SSL02_top100.txt
Written 100 entries to ../outputs/topk/Llama3.1_FSL16_top100.txt
Written 100 entries to ../outputs/topk/bad_Llama3.1_FSL16_top100.txt


In [1]:
import numpy as np
import pickle, os

train_context = np.load(os.path.join('../data/ED', 'sys_dialog_texts.train.npy'),  allow_pickle=True)
train_target  = np.load(os.path.join('../data/ED', 'sys_target_texts.train.npy'),  allow_pickle=True)

/home/new/miniconda3/lib/python3.13/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [17]:
from sentence_transformers import SentenceTransformer, util
import torch

# Load a fast, lightweight embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

target_phrases = [
    "that is so cool",
    "i am sorry to hear that",
    "that is even more",
    "oh my",
    "that is",
]
target_embeddings = model.encode(target_phrases, convert_to_tensor=True)

def contains_phrase_sliding_window(text, window_size=5, threshold=0.75):
    words = text.split()
    
    # If the text is shorter than the window, just embed the whole thing
    if len(words) <= window_size:
        chunks = [text]
    else:
        # Create overlapping chunks of 'window_size' words
        chunks = []
        for i in range(len(words) - window_size + 1):
            chunk = " ".join(words[i:i + window_size])
            chunks.append(chunk)
            
    # Embed all chunks
    chunk_embeddings = model.encode(chunks, convert_to_tensor=True)
    
    # Compute cosine similarity between all chunks and all target phrases
    # Resulting matrix shape: (num_chunks, num_targets)
    cosine_scores = util.cos_sim(chunk_embeddings, target_embeddings)
    
    # Find the highest similarity score
    max_score = torch.max(cosine_scores).item()
    chunk_idx, target_idx = torch.where(cosine_scores == max_score)
    
    if max_score >= threshold:
        # Optional: find exactly which phrase and chunk matched
        chunk_idx, target_idx = torch.where(cosine_scores == max_score)
        # matched_chunk = chunks[chunk_idx[0]]
        # matched_target = target_phrases[target_idx[0]]
        # print(f"Match found! Chunk: '{matched_chunk}' matched Target: '{matched_target}' (Score: {max_score:.2f})")
        return True, max_score, target_idx[0].item()
        
    return False, max_score, 0.0

# Test it
sentence = "Well, I feel so glad to hear that, anyway let's move on to the next task."
print(contains_phrase_sliding_window(sentence))

(False, 0.5118780732154846, 0.0)


In [19]:
get_in_sentence = list()
target_list = [0]*len(target_phrases)
len_target = len(train_target)

for sentence in train_target[:int(len_target*0.2)]:
    simlarity, score, selected_phrase = contains_phrase_sliding_window(sentence)
    if simlarity:
        get_in_sentence.append((simlarity, score, selected_phrase))
        target_list[int(selected_phrase)] += 1
    
print(f"Total simlarity number of target text: {len(get_in_sentence)}, compare with overall section length: {len(get_in_sentence) / len(train_target[:int(len_target*0.2)]):.2f}")
for idx, phrase in enumerate(target_phrases):
    print(f"Phrase: '{phrase}' - Count in Target Text: {target_list[idx]}")

Total simlarity number of target text: 210, compare with overall section length: 0.03
Phrase: 'that is so cool' - Count in Target Text: 9
Phrase: 'i am sorry to hear that' - Count in Target Text: 201
Phrase: 'that is even more' - Count in Target Text: 0
Phrase: 'oh my' - Count in Target Text: 0
Phrase: 'that is' - Count in Target Text: 0


In [18]:
print(f"compare with overall section length: {len(get_in_sentence) / len(train_target[:int(len_target*0.2)]):.2f}")

compare with overall section length: 0.03
